# பாடம் 11 - மாதிரி.context ஒப்பந்தம் (MCP)

**மாதிரி.context ஒப்பந்தம் (MCP)** என்பது ஒரு திறந்த தரநிலை ஆகும், இது முகவர்களுக்கு இயங்கு நேரத்தில் கருவிகள், வளங்கள் மற்றும் தரவுத் தரவுத்தளங்களை டைனமிக் முறையில் கண்டுபிடித்து பயன்படுத்த அனுமதிக்கிறது. முகவரியில் கருவிகளை கடுமையாக நிரலாக்குவதற்கு பதிலாக, MCP முகவர்களுக்கு தேவையான திறன்களை வெளிப்படுத்தும் வெளி சேவையகங்களுடன் இணைக்க அனுமதிக்கிறது.

இந்த பாடத்தில், நீங்கள் கற்றுக்கொள்ள உள்ளீர்கள்:
- MCP என்றால் என்ன மற்றும் முகவர் அமைப்புகளுக்கு அது ஏன் முக்கியம்
- MCP கிளையண்ட்-சேவையக கட்டமைப்பு எப்படி செயல்படுகிறது
- MCP முறையில் கருவி கண்டுபிடிப்பை பயன்படுத்தும் முகவர்களை எப்படி உருவாக்குவது


## அமைப்பு

**முன் தேவைகள்:**
- முன்னிலையில் மாடல் கொண்டு Microsoft Foundry திட்டம்
- அங்கீகாரத்திற்கு `az login` இயக்கவும்


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## மாடல் Context சீர்குலைப்பு (MCP) என்ன?

MCP என்பது ஏ.ஐ. முகவரிகள் வெளிப்புற கருவிகள் மற்றும் தரவுத்தளங்களுடன் கண்டறிந்து தொடர்பு கொள்ள ஒரு நிலையான முறையை வரையறுக்கிறது:

- **MCP சேவையகம்**: ஒரு நிலையான சீர்குலைப்பின் மூலம் கருவிகள், வளங்கள் மற்றும் வழிகாட்டுதல்களை வெளிப்படுத்துகிறது
- **MCP கிளையண்ட்**: சேவையகத்துடன் இணைந்து கிடைக்கும் திறன்களை கண்டறியும் முகவர் இயக்கநேரம்
- **இயங்கக் கண்டறிதல்**: முகவர்கள் கடினமாக குறியிடப்பட்ட கருவிகள் தேவையில்லை - அவர்கள் இயக்கநேரத்தில் கிடைக்கும் ஒன்றை கண்டறிந்து கொள்கிறார்கள்

இது முகவர் குறியீட்டை மாற்றாமை புதிய திறன்களை சேர்க்கக்கூடிய விரிவாக்கக்கூடிய முகவர் அமைப்புகளை உருவாக்குவதற்கு சக்திவாய்ந்தது.


## MCP எப்படி செயல்படுகிறது

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. முகவர் (MCP கிளையண்ட்) ஒரு MCP சேவையகத்துடன் இணைக்கிறது
2. சேவையகம் கிடைக்கும் கருவிகளின் பட்டியலும் அவற்றின் விதிகளும் கொண்ட பதிலை அனுப்புகிறது
3. முகவர் அதன் காரணீகத்தில் கண்டறிந்த எந்த கருவியையும் அழைக்கலாம்
4. முடிவுகள் அதே செயல்முறை வழியாக திரும்பி வருகிறது


## MCP கருவி கண்டுபிடிப்பை பிணைக்குதல்

உண்மையான MCP சேவையகம் ஒரு இயங்கும் சேவையக செயல்முறையை தேவையாக்கும் காரணத்தினால், MCP-க்கு இணைக்கப்பட்ட இடைமுக சேவை வழங்கும் செயல்பாடுகளை பிணைக்கும் `@tool` செயல்பாடுகளைப் பயன்படுத்தி சரிபார்ப்போம்.

உற்பத்தியில், இந்த கருவிகள் உள்ளூர் வரையறுக்கப்படுவதைவிட MCP சேவையகத்திலிருந்து ஊடுருவிக் கண்டுபிடிக்கப்படும்.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-மாதிரியான கருவிகளுடன் ஒரு முகவரியை கட்டமைத்தல்


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## உற்பத்தியில் MCP

உற்பத்தி சூழலில், MCP சக்திவாய்ந்த மாதிரிகள் மூலம் செயல்படுகிறது:

- **தொடர்ச்சி கருவி கண்டறிதல்**: முகவர்கள் MCP சர்வர்களுடன் இணைத்து இயக்க நேரத்தில் கருவிகளை கண்டறிகின்றார்கள்
- **தனித்தனி கட்டமைப்பு**: கருவி வழங்குநர்கள் முகவருடன் தனி தனியாக புதுப்பிக்க முடியும்
- **கழிவு நிறுவனர் பகிர்வு**: குழுக்கள் எந்த முகவரும் பயன்படுத்தக்கூடிய MCP சர்வர்களின் மூலம் திறன்களை வெளிப்படுத்த முடியும்
- **Microsoft முகவர் கட்டமைப்பு ஆதரவு**: MAF இல் `mcp` ஒருங்கிணைப்பின் மூலம் உள்ளமைக்கப்பட்ட MCP கிளையன்ட் ஆதரவு உள்ளது

இயல்பான MCP சர்வரை MAF உடன் பயன்படுத்த, `hosted_mcp_tool()` அல்லது MCP கிளையன்ட் ஒருங்கிணைப்பின் மூலம் நீங்கள் இணைகிறீர்கள்.

**மேலும் அறிய:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
